# Cardiac LGE Clustering — Model Comparison

This notebook runs inference using all pre-trained models on the Cardiac LGE scar mapping dataset and compares their clustering performance.

**Models**: Baseline GMM, VAE-GMM, Diffusion-VAE, Ours  
**Metrics**: Accuracy (Hungarian), NMI, ARI  
**Clusters**: K=5 pathologies (Healthy, Ischemic Subendo, Ischemic Transmural, DCM Mid-wall, Myocarditis Epicardial)

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from notebook_utils import (
    run_all_models, build_metrics_table, plot_cluster_means_grid,
    plot_all_models_cluster_samples, compute_cluster_purity_table,
    save_results_cache, MODEL_DISPLAY_NAMES,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Data & Run All Models

We evaluate on the cardiac test set (20% holdout, 128x128 LGE images). For each model we extract features, fit a GMM with K=5, and assign clusters.

For Diffusion-VAE, we use denoised latents at t=2 (best performing timestep).

In [ ]:
MODEL_ORDER = ['baseline_gmm', 'vae_gmm', 'diffusion_vae', 'clast']

print('Running inference on all models...')
results, (test_loader, dataset_info) = run_all_models('cardiac', MODEL_ORDER, device)
print(f'\nModels evaluated: {[results[m]["display_name"] for m in MODEL_ORDER if m in results]}')

# Cache cluster assignments
save_results_cache(results, 'cardiac')

## 2. Performance Metrics

Comparing clustering performance across all models using Hungarian accuracy, NMI, and ARI.

In [ ]:
metrics_df = build_metrics_table(results, supervised=True)
metrics_df

## 3. Cluster Means Comparison

Each row shows the 5 cluster prototypes for one model. Neural models decode from latent space; the baseline uses pixel-space centroids.

In [ ]:
plot_cluster_means_grid(results, MODEL_ORDER, device, title='Cardiac — Cluster Means')

## 4. Cluster Purity

Per-cluster breakdown for each model, showing which pathology class dominates and the purity ratio.

In [ ]:
for mname in MODEL_ORDER:
    if mname not in results:
        continue
    r = results[mname]
    print(f"\n{'='*60}")
    print(f"{r['display_name']}")
    print(f"{'='*60}")
    purity_df = compute_cluster_purity_table(
        r['labels'], r['cluster_labels'], r['num_clusters'], dataset_info['class_names']
    )
    display(purity_df.style.format({'Purity': '{:.2%}'}).background_gradient(
        subset=['Purity'], cmap='Greens', vmin=0, vmax=1
    ))

## 5. Cluster Samples

Change `CLUSTER_IDX` below and re-run the cell to browse samples from different clusters.

In [ ]:
CLUSTER_IDX = 0  # <-- Change this value (0-4) and re-run

plot_all_models_cluster_samples(results, MODEL_ORDER, CLUSTER_IDX, n_samples=8)

## Analysis

**Key observations:**

- **Baseline GMM** in pixel space (16,384D) struggles with the high-dimensional cardiac images. Cluster means are extremely blurry and purity is low.

- **VAE-GMM** compresses images to a 16D latent space where GMM clustering separates pathologies more effectively. Decoded cluster means show recognizable cardiac patterns.

- **Diffusion-VAE** refines the latent space with a denoiser (best at t=2). This can sharpen the cluster boundaries in the latent space.

- **Ours** achieves the highest performance with manifold-aware clustering. The heat-kernel medoid selection ensures cluster prototypes remain on the data manifold, producing the sharpest and most clinically interpretable prototypes.